# Phase 6.5C.1 — Failed-study diagnostics (zero LLM calls)
Original run prepared RV546 (3/9). RESP30X-EBA and B3461058 failed with ValueError; previous summary omitted messages and stack traces. This diagnostic uses the identical frozen commit, exact original PDF SHA256 and page intervals, and unchanged chunk/retrieval policy. No gold labels or model requests. It records the first failing stage for each study, tokenizer limits and any oversized chunks, without truncating text or changing the algorithm. No need to redownload or rerun RV546.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib, json, shutil, time, re, hashlib
from datetime import datetime, timezone
REPO='https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH='simplified-rag-v3'
PIN='67cc048b69369f1fd12b1eccd131c454e762a71a'
ROOT=Path('/content/tfl-phase65c-frozen')
if not ROOT.exists():
    subprocess.run(['git','clone','--branch',BRANCH,REPO,str(ROOT)],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','--detach',PIN],check=True)
assert subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()==PIN
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[retrieval,test]'],check=True)
SRC=ROOT/'src'
sys.path.insert(0,str(SRC))
for name in list(sys.modules):
    if name=='tfl_agent' or name.startswith('tfl_agent.'):
        del sys.modules[name]
for p in (SRC/'tfl_agent').rglob('__pycache__'):
    shutil.rmtree(p)
importlib.invalidate_caches()
import tfl_agent
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC.resolve())
from tfl_agent.agent.unified_pipeline import TFLAgent, ResponseExecutor, MATCH_SYSTEM, sha_file, write_json, digest
from tfl_agent.agent.phase6_simplified_replay import APP, DISC
assert all('"'+x+'"' in MATCH_SYSTEM for x in APP|DISC|{'supported','uncertain','incompatible','sap','title'})
subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests/test_unified_pipeline.py')],check=True)
print('Frozen commit:',PIN)

In [ ]:
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive',timeout_ms=180000)
BASE=Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo')
DATA=BASE/'Phase65C_Heldout_SAPs'
DEST=BASE/'Phase65C_Diagnostics_Outputs'
DEST.mkdir(parents=True,exist_ok=True)
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT=Path('/content/Phase65C_Diagnostics')/RUN_ID
OUT.mkdir(parents=True,exist_ok=False)
STUDIES=[
 {'study_id':'RESP30X-EBA','file':'NCT06041919_SAP_001.pdf','sha':'a761ce97ee702acef9a457bb5b39c4d2a82eab5009885cec9f4f562a1887e93e',
  'pages':(1,36),'entry':'14.1.2','title':'Demographic Data'},
 {'study_id':'B3461058','file':'NCT03842163_SAP_001.pdf','sha':'ea04be230aa6d79e40bbc2cef6a1048b7915ae8e9f5d3077cf8177959be11a6e',
  'pages':(1,17),'entry':'15.1.1','title':'Number and Percentage of Patients in Each Analysis Set'}
]
from tfl_agent.tools.sap_hybrid import load_sap_body, LocalEncoder, HybridSAPIndex
from tfl_agent.tools.sap_scope import segment_blocks, scoped_chunks
from tfl_agent.agent.unified_pipeline import source_records
import torch, traceback, platform
from importlib.metadata import version, PackageNotFoundError
def installed(name):
    try: return version(name)
    except PackageNotFoundError: return None
encoder=LocalEncoder(device='cuda' if torch.cuda.is_available() else 'cpu')
tok=encoder.model.tokenizer
env={'commit':PIN,'python':platform.python_version(),'device':str(encoder.model.device),
     'encoder':encoder.model_name,'max_seq_length':encoder.max_seq_length,
     'versions':{x:installed(x) for x in ['pymupdf','sentence-transformers','transformers','torch','numpy','rank-bm25']}}
write_json(OUT/'Environment.json',env)
print(json.dumps(env,indent=2))
executor=ResponseExecutor()
agent=TFLAgent(ROOT/'generic_template_library',encoder=encoder,executor=executor)
rows=[]
for s in STUDIES:
    row={'study_id':s['study_id'],'body_pages':list(s['pages']),'target':s['title'],'stages':[]}
    stage='verify_pdf'
    start=time.perf_counter()
    def passed(name,**kwargs):
        row['stages'].append({'stage':name,'status':'passed',**kwargs})
        print(s['study_id'],name,kwargs,flush=True)
    try:
        path=DATA/s['file']
        actual=sha_file(path)
        if actual!=s['sha']:
            raise ValueError('Original SAP SHA256 mismatch')
        passed(stage,sha256=actual)
        stage='load_sap_body'
        sap=load_sap_body(path,s['pages'])
        passed(stage,blocks=len(sap['blocks']))
        stage='segment_blocks'
        segments,excluded,headings=segment_blocks(sap['blocks'])
        passed(stage,segments=len(segments))
        stage='scoped_chunks'
        chunks,metadata=scoped_chunks(segments)
        passed(stage,chunks=len(chunks))
        stage='source_records'
        records=source_records(sap['blocks'],segments,chunks,metadata)
        passed(stage,records=len(records))
        stage='tokenizer_diagnostics'
        lengths=[]
        for c in chunks:
            n=len(tok.encode(c.text,add_special_tokens=True,truncation=False))
            lengths.append({'chunk_id':c.chunk_id,'pages':list(c.pages),'words':len(c.text.split()),'tokens':n,'characters':len(c.text)})
        oversized=[x for x in lengths if x['tokens']>encoder.max_seq_length]
        write_json(OUT/f"{s['study_id']}_Token_Lengths.json",lengths)
        row['tokenizer']={'limit':encoder.max_seq_length,'maximum':max((x['tokens'] for x in lengths),default=0),
                          'oversized':oversized}
        passed(stage,maximum=row['tokenizer']['maximum'],oversized_count=len(oversized))
        stage='HybridSAPIndex'
        # Deliberately retain the original validation; do not bypass oversized-chunk checks.
        index=HybridSAPIndex(chunks,encoder)
        passed(stage)
        stage='agent_prepare'
        agent._study_cache[(s['sha'],tuple(s['pages']))]=(records,index,excluded,headings,len(sap['blocks']))
        plan=agent.prepare(path,s['title'],body_pages=s['pages'],study_id=s['study_id'],entry_id=s['entry'])
        passed(stage,primary=len(plan['retrieval']['primary_ids']),supplementary=len(plan['retrieval']['supplementary_ids']))
        stage='complete'
        row['status']='prepared'
    except Exception as exc:
        row.update(status='failed',failed_stage=stage,error_type=type(exc).__name__,error_message=str(exc),traceback=traceback.format_exc())
        print(s['study_id'],stage,type(exc).__name__,str(exc),flush=True)
        print(row['traceback'])
    finally:
        row['seconds']=round(time.perf_counter()-start,3)
        rows.append(row)
        write_json(OUT/'Diagnostics.json',rows)
assert executor.new_attempts==0
summary={'commit':PIN,'new_attempts':0,'studies':len(rows),'failed':sum(x['status']=='failed' for x in rows),
         'results':[{k:v for k,v in x.items() if k!='traceback'} for x in rows]}
write_json(OUT/'Summary.json',summary)
saved=Path(shutil.make_archive(str(DEST/RUN_ID),'zip',root_dir=OUT))
print('Saved:',saved)
print(json.dumps(summary,indent=2,ensure_ascii=False))

Return the ZIP under Phase65C_Diagnostics_Outputs. Do not change PDF ranges, skip validation, replace cases or enable LLM inference. If tokenizer limits are the blocker, implement a general provenance-preserving token-aware split on development/synthetic tests, freeze a new revision and rerun the whole cohort consistently. No inference predictions have been generated here; any change must be documented before evaluation.